# 00 - Qwen2.5-7B-Instruct 4-bit smoke test (Kaggle / Colab)

**Mục đích.** Kiểm tra sớm, trước khi xây RAG, rằng Qwen2.5-7B-Instruct ở dạng 4-bit (NF4) chạy được trên GPU T4 và đo bốn đại lượng làm cơ sở cho các quyết định kỹ thuật:

| Đại lượng | Dùng để quyết định |
|---|---|
| VRAM sau khi nạp mô hình và VRAM đỉnh khi sinh | Có đủ chỗ cho BGE-M3 và PhoBERT trong cùng phiên không; có phải chuyển sang bản 3B không |
| Độ trễ và tốc độ sinh (token/giây) | Số đoạn truy hồi k tối đa chấp nhận được (k ∈ {3, 5, 10}) |
| Hành vi trả lời chỉ dựa trên ngữ cảnh (17 câu) | Prompt v1 có làm mô hình từ chối đúng khi ngữ cảnh thiếu thông tin hay không |
| Tỷ lệ câu trả lời lẫn ký tự CJK (chữ Hán) | Có cần cơ chế kiểm tra và sinh lại hay không |

**Phạm vi.** Bộ 17 câu trong `smoke_test.jsonl` chỉ phục vụ kiểm tra kỹ thuật. Bộ này **không** thuộc tập 100 câu đánh giá chính thức, và các con số thu được ở đây không được đưa vào bảng kết quả của báo cáo. Riêng thông số VRAM, độ trễ và phiên bản thư viện có thể trích dẫn làm thông tin môi trường.

**Cách chạy trên Kaggle**
1. Tạo Dataset mới từ thư mục `kaggle/smoke_test_dataset/` (hai file `smoke_test.jsonl` và `long_context.txt`).
2. Tạo Notebook mới, chọn *File > Import notebook* và nạp file này.
3. Bên phải: *Add Input* để gắn Dataset ở bước 1. *Session options > Accelerator*: **GPU T4 x2** (mã dưới đây chỉ dùng GPU 0, tương đương một T4 của Colab). *Internet*: **On** (cần để tải trọng số từ Hugging Face).
4. Chạy lần lượt từ trên xuống (*Run All*). Lần đầu phải tải khoảng 15 GB trọng số nên tốn vài phút.
5. Tải hai file kết quả trong `/kaggle/working/` (`smoke_test_results.json`, `smoke_test_items.csv`) về và lưu vào repo.

## 1. Thiết lập môi trường

- `CUDA_VISIBLE_DEVICES=0` ép chỉ dùng một GPU, để số đo VRAM phản ánh đúng một T4 (15 GB) và mô hình không bị chia sang GPU thứ hai.
- `HF_HOME` trỏ vào ổ tạm của Kaggle để bộ nhớ đệm trọng số (khoảng 15 GB) không làm đầy `/kaggle/working` (giới hạn 20 GB và sẽ bị đóng gói vào output).
- `bitsandbytes` là thư viện thực hiện lượng tử hóa 4-bit. Cell này chỉ cập nhật `bitsandbytes` và `accelerate`; `transformers` và `torch` giữ nguyên phiên bản có sẵn của Kaggle.

In [ ]:
import os

# Use a single GPU so that the VRAM numbers correspond to one T4.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Keep the ~15 GB weight cache out of /kaggle/working.
if os.path.isdir("/kaggle/temp"):
    os.environ["HF_HOME"] = "/kaggle/temp/hf"

!pip install -q -U bitsandbytes accelerate

In [ ]:
import csv
import glob
import importlib.metadata as md
import json
import platform
import re
import statistics
import time
import unicodedata
from pathlib import Path

import torch
import transformers

assert torch.cuda.is_available(), "No GPU found. Set Accelerator to GPU T4 in the session options."

props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu_name": props.name,
    "gpu_total_gb": round(props.total_memory / 1e9, 2),
    "compute_capability": f"{props.major}.{props.minor}",
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "accelerate": md.version("accelerate"),
    "bitsandbytes": md.version("bitsandbytes"),
}
print(json.dumps(ENV, indent=2))

## 2. Nạp dữ liệu thử nghiệm

`smoke_test.jsonl` có 17 dòng. Mỗi dòng gồm: câu hỏi tiếng Việt, đoạn ngữ cảnh, cờ `answerable` (ngữ cảnh có chứa đáp án hay không) và `gold_contains` (các chuỗi dự kiến xuất hiện trong đáp án đúng).

| Nhóm | Số câu | Kỳ vọng |
|---|---|---|
| Ngữ cảnh tiếng Anh (trích từ hai PDF Galaxy S24), có đáp án | 10 | Trả lời bằng tiếng Việt, đúng số liệu |
| Ngữ cảnh tiếng Anh, không có đáp án (gồm 1 câu hỏi về S24 **Ultra** trong khi ngữ cảnh chỉ nói về S24) | 4 | Từ chối |
| Ngữ cảnh tiếng Việt, có đáp án | 2 | Trả lời đúng |
| Ngữ cảnh tiếng Việt, không có đáp án | 1 | Từ chối |

Cấu hình này mô phỏng đặc điểm của đề tài: câu hỏi tiếng Việt nhưng tài liệu miền ứng dụng bằng tiếng Anh.

In [ ]:
candidates = (
    glob.glob("/kaggle/input/**/smoke_test.jsonl", recursive=True)
    + glob.glob("./smoke_test.jsonl")
    + glob.glob("./**/smoke_test.jsonl", recursive=True)
)
assert candidates, "smoke_test.jsonl not found. Attach the dataset via 'Add Input' (Kaggle) or upload it next to the notebook (Colab)."
DATA_DIR = Path(candidates[0]).parent
print("data dir:", DATA_DIR)

with open(DATA_DIR / "smoke_test.jsonl", encoding="utf-8") as f:
    ITEMS = [json.loads(line) for line in f if line.strip()]
LONG_TEXT = (DATA_DIR / "long_context.txt").read_text(encoding="utf-8")

print(len(ITEMS), "items;", sum(i["answerable"] for i in ITEMS), "answerable;", sum(not i["answerable"] for i in ITEMS), "unanswerable")
print(ITEMS[0]["question"])

## 3. Nạp Qwen2.5-7B-Instruct ở dạng 4-bit

Cấu hình lượng tử hóa:

- `load_in_4bit=True`: trọng số lưu bằng 4 bit thay vì 16 bit, giảm bộ nhớ khoảng 4 lần (từ khoảng 15 GB xuống khoảng 5-6 GB; con số thực tế được đo ở cell này).
- `bnb_4bit_quant_type="nf4"`: kiểu số NormalFloat4, phù hợp với phân phối xấp xỉ chuẩn của trọng số mạng nơ-ron.
- `bnb_4bit_use_double_quant=True`: lượng tử hóa cả các hằng số tỷ lệ, tiết kiệm thêm một ít bộ nhớ.
- `bnb_4bit_compute_dtype=torch.float16`: phép tính thực hiện bằng float16 vì T4 không hỗ trợ bfloat16.
- `device_map={"": 0}`: đặt toàn bộ mô hình lên GPU 0.

Trọng số gốc (bf16) phải được tải về đủ rồi mới lượng tử hóa khi nạp, nên bước này chiếm nhiều thời gian và dung lượng đĩa nhất.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    # Newer transformers versions name the argument `dtype`.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map={"": 0}, dtype=torch.float16
    )
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map={"": 0}, torch_dtype=torch.float16
    )
model.eval()
torch.cuda.synchronize()

LOAD = {
    "model_id": MODEL_ID,
    "load_seconds": round(time.perf_counter() - t0, 1),
    "model_footprint_gb": round(model.get_memory_footprint() / 1e9, 2),
    "vram_allocated_after_load_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
    "vram_peak_during_load_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
}
print(json.dumps(LOAD, indent=2))

## 4. Prompt v1 và hàm sinh câu trả lời

Prompt hệ thống áp dụng các ràng buộc của kế hoạch (mục 3.5e):

1. chỉ dựa trên ngữ cảnh, không dùng kiến thức ngoài;
2. trả lời bằng tiếng Việt;
3. khi ngữ cảnh thiếu thông tin, trả lời đúng một câu cố định ("Không tìm thấy thông tin trong tài liệu."), để việc nhận diện từ chối có thể thực hiện bằng so khớp chuỗi.

Giải mã **tham lam** (`do_sample=False`) để kết quả lặp lại được giữa các lần chạy. Mỗi lần gọi ghi lại: số token đầu vào, số token sinh, thời gian, token/giây và VRAM đỉnh.

Các hàm kiểm tra:

- `has_cjk`: phát hiện ký tự Hán/Nhật/Hàn trong câu trả lời (Qwen thỉnh thoảng chuyển sang tiếng Trung).
- `is_refusal`: nhận diện câu từ chối.
- `gold_hit`: kiểm tra có chuỗi nào trong `gold_contains` xuất hiện trong đáp án hay không (chỉ là chỉ báo gần đúng; mọi trường hợp "không khớp" cần đọc lại bằng mắt).

In [ ]:
SYSTEM_PROMPT = (
    "Bạn là trợ lý hỏi đáp tiếng Việt. "
    "Chỉ trả lời dựa trên phần NGỮ CẢNH được cung cấp, không dùng kiến thức bên ngoài. "
    "Luôn trả lời bằng tiếng Việt, ngắn gọn, chỉ dùng chữ Latinh (không dùng chữ Hán). "
    "Nếu ngữ cảnh không chứa thông tin để trả lời, chỉ trả lời đúng một câu: "
    "'Không tìm thấy thông tin trong tài liệu.'"
)

CJK_RE = re.compile(r"[\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff\uac00-\ud7af]")


def norm(s: str) -> str:
    return unicodedata.normalize("NFC", s).lower()


def has_cjk(text: str) -> bool:
    return bool(CJK_RE.search(text))


def is_refusal(text: str) -> bool:
    return "không tìm thấy thông tin" in norm(text)


def gold_hit(text: str, gold: list) -> bool:
    return any(norm(g) in norm(text) for g in gold)


@torch.inference_mode()
def generate(context: str, question: str, max_new_tokens: int = 256) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"NGỮ CẢNH:\n{context}\n\nCÂU HỎI: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    n_in = inputs["input_ids"].shape[1]

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=False, temperature=None, top_p=None, top_k=None,
    )
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0

    new_ids = out[0, n_in:]
    n_out = int(new_ids.shape[0])
    return {
        "answer": tokenizer.decode(new_ids, skip_special_tokens=True).strip(),
        "prompt_tokens": n_in,
        "new_tokens": n_out,
        "seconds": round(dt, 2),
        "tokens_per_second": round(n_out / dt, 2) if dt > 0 else None,
        "vram_peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


# Warm-up call (CUDA kernels and caches are initialised on first use and would distort timings).
_ = generate("Galaxy S24 có màn hình 6.2 inch.", "Màn hình bao nhiêu inch?", max_new_tokens=16)
print("warm-up done")

## 5. Chạy 17 câu thử nghiệm

Mỗi câu được gán một trong năm trạng thái, dựa trên cờ `answerable` và kết quả kiểm tra tự động:

| Trạng thái | Điều kiện | Ý nghĩa |
|---|---|---|
| `correct` | có đáp án, không từ chối, khớp `gold_contains` | đạt |
| `check_manually` | có đáp án, không từ chối, không khớp | cần đọc lại (có thể đúng nhưng diễn đạt khác) |
| `false_refusal` | có đáp án nhưng từ chối | từ chối sai |
| `correct_refusal` | không có đáp án và từ chối | từ chối đúng |
| `possible_hallucination` | không có đáp án nhưng vẫn trả lời | nghi ảo giác, cần đọc lại |

In [ ]:
def status_of(item: dict, res: dict) -> str:
    refused = is_refusal(res["answer"])
    if item["answerable"]:
        if refused:
            return "false_refusal"
        return "correct" if gold_hit(res["answer"], item["gold_contains"]) else "check_manually"
    return "correct_refusal" if refused else "possible_hallucination"


ROWS = []
for item in ITEMS:
    res = generate(item["context"], item["question"])
    row = {
        "id": item["id"],
        "lang_context": item["lang_context"],
        "answerable": item["answerable"],
        "question": item["question"],
        "answer": res["answer"],
        "status": status_of(item, res),
        "has_cjk": has_cjk(res["answer"]),
        "prompt_tokens": res["prompt_tokens"],
        "new_tokens": res["new_tokens"],
        "seconds": res["seconds"],
        "tokens_per_second": res["tokens_per_second"],
        "vram_peak_gb": res["vram_peak_gb"],
    }
    ROWS.append(row)
    print(f"[{row['status']:>22}] {row['id']}: {row['answer'][:110]!r}")

In [ ]:
from collections import Counter

status_counts = Counter(r["status"] for r in ROWS)
SUMMARY = {
    "n_items": len(ROWS),
    "status_counts": dict(status_counts),
    "cjk_rate": round(sum(r["has_cjk"] for r in ROWS) / len(ROWS), 3),
    "median_seconds": statistics.median(r["seconds"] for r in ROWS),
    "median_tokens_per_second": statistics.median(r["tokens_per_second"] for r in ROWS if r["tokens_per_second"]),
    "max_vram_peak_gb": max(r["vram_peak_gb"] for r in ROWS),
}
print(json.dumps(SUMMARY, indent=2, ensure_ascii=False))

## 6. Thử tải theo độ dài ngữ cảnh

Mục 3.3 của kế hoạch so sánh k ∈ {3, 5, 10} đoạn đưa vào LLM. Nếu mỗi đoạn dài khoảng 200-300 token (giả định, chưa chốt), ngữ cảnh tương ứng vào khoảng 0.6k, 1.5k và 3k token. Cell này ghép văn bản của hai PDF (lặp lại nếu cần) thành ngữ cảnh dài đúng 512, 1536, 3072 và 6144 token, hỏi một câu cố định và ghi lại độ trễ cùng VRAM đỉnh. Văn bản lặp chỉ dùng để đo tải; không đánh giá chất lượng câu trả lời ở bước này.

Kết quả `OOM` nghĩa là hết bộ nhớ GPU ở độ dài đó.

In [ ]:
long_ids = tokenizer(LONG_TEXT, add_special_tokens=False)["input_ids"]


def padded_context(n_tokens: int) -> str:
    ids = []
    while len(ids) < n_tokens:
        ids += long_ids
    return tokenizer.decode(ids[:n_tokens])


LENGTH_TEST = []
for n in (512, 1536, 3072, 6144):
    try:
        res = generate(padded_context(n), "Galaxy S24 nặng bao nhiêu gram?", max_new_tokens=64)
        LENGTH_TEST.append({
            "target_context_tokens": n, "prompt_tokens": res["prompt_tokens"],
            "seconds": res["seconds"], "tokens_per_second": res["tokens_per_second"],
            "vram_peak_gb": res["vram_peak_gb"], "oom": False,
        })
    except torch.cuda.OutOfMemoryError:
        LENGTH_TEST.append({"target_context_tokens": n, "oom": True})
        torch.cuda.empty_cache()
    print(LENGTH_TEST[-1])

## 7. Lưu kết quả

Hai file được ghi vào `/kaggle/working/` (Kaggle) hoặc thư mục hiện tại (Colab):

- `smoke_test_results.json`: môi trường, thông số nạp mô hình, tóm tắt, kết quả thử tải theo độ dài, kèm prompt đã dùng.
- `smoke_test_items.csv`: kết quả từng câu (để đọc lại các trường hợp `check_manually` và `possible_hallucination`).

In [ ]:
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")

RESULTS = {
    "env": ENV,
    "load": LOAD,
    "summary": SUMMARY,
    "length_test": LENGTH_TEST,
    "system_prompt": SYSTEM_PROMPT,
}
(WORK / "smoke_test_results.json").write_text(json.dumps(RESULTS, indent=2, ensure_ascii=False), encoding="utf-8")

with open(WORK / "smoke_test_items.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(ROWS[0].keys()))
    writer.writeheader()
    writer.writerows(ROWS)

print("saved to", WORK.resolve())
print(json.dumps(RESULTS["load"], indent=2))
print(json.dumps(RESULTS["summary"], indent=2, ensure_ascii=False))

## 8. Cách đọc kết quả

Các ngưỡng dưới đây là **đề xuất** để có căn cứ ra quyết định, không phải yêu cầu của đề bài hay của kế hoạch; có thể điều chỉnh sau khi xem số liệu thực.

| Quan sát | Hệ quả đề xuất |
|---|---|
| Nạp thành công, `vram_allocated_after_load_gb` còn chỗ cho BGE-M3 (khoảng 1-2 GB) và PhoBERT-base | Giữ Qwen2.5-7B làm LLM chính |
| `OOM` ở 3072 token | Giới hạn k ≤ 5 trong cấu hình RAG, hoặc rút ngắn đoạn |
| Độ trễ trung vị ở 1536 token quá cao so với yêu cầu demo (ví dụ trên 30 giây/câu) | Giảm k hoặc cân nhắc Qwen2.5-3B (kiểm tra giấy phép của bản 3B trước khi dùng, theo kế hoạch) |
| `cjk_rate` > 0 | Giữ cơ chế kiểm tra CJK và sinh lại đã nêu trong kế hoạch |
| Nhiều `false_refusal` hoặc `possible_hallucination` | Ghi nhận làm cơ sở sửa prompt v1 (việc chỉnh prompt chỉ thực hiện trên 30 câu dev, không dựa vào 70 câu test) |

Sau khi chạy, các phiên bản thư viện in ở cell đầu là căn cứ để chốt lại `requirements.txt`.